# Tokens, token IDs и embeddings

Цель: проследить путь `текст → токены → token IDs → embedding vectors` и уверенно объяснять формы `(batch, sequence_length, embedding_dimension)`.

## 1. Гипотеза до запуска кода

Ответь своими словами:

1. Чем token отличается от token ID?
2. Где хранятся embedding-векторы: в tokenizer или в модели?
3. Что означают три измерения `(batch, sequence_length, embedding_dimension)`?
4. Если `vocab_size=10` и `embedding_dimension=4`, сколько параметров содержит embedding-слой?

**Мои ответы:**

1. Token — элемент, выделенный tokenizer: слово, часть слова, символ или байтовый фрагмент. Token ID — целочисленный индекс этого токена в словаре tokenizer.
2. Обучаемые embedding-векторы хранятся в embedding-слое модели; tokenizer сопоставляет токены и их IDs.
3. `batch` — количество последовательностей, обрабатываемых одновременно; `sequence_length` — число позиций в каждой последовательности данного тензора; `embedding_dimension` — длина вектора одной позиции.
4. $10 \cdot 4 = 40$ обучаемых параметров.

In [1]:
import torch
from torch import nn

torch.manual_seed(42)

## 2. Учебный tokenizer

В реальных LLM tokenizer может разбивать текст на слова, части слов, символы или байтовые фрагменты. Здесь используем готовые токены, чтобы сосредоточиться на формах. Tokenizer сопоставляет токену целочисленный ID; он не хранит обучаемые semantic embeddings модели.

In [4]:
vocabulary = {
    "<pad>": 0,
    "я": 1,
    "изучаю": 2,
    "трансформеры": 3,
    "люблю": 4,
}

sentences = [
    ["я", "изучаю", "трансформеры"],
    ["я", "люблю", "трансформеры"],
]

# Преобразуем каждый токен в ID с помощью vocabulary.
# Ожидаемая форма token_ids: (2, 3).
token_ids = torch.tensor([
    [vocabulary[token] for token in sentence]
    for sentence in sentences
])
print(token_ids)
print(token_ids.shape)

tensor([[1, 2, 3],
        [1, 4, 3]])
torch.Size([2, 3])


**Наблюдение:** что означают строки и столбцы `token_ids`?

Каждая строка соответствует одной последовательности в batch, а каждый столбец — позиции токена внутри последовательности. Значение в ячейке — token ID, а не числовой признак токена.

## 3. Embedding lookup

`nn.Embedding(vocab_size, embedding_dim)` хранит обучаемую таблицу формы `(vocab_size, embedding_dim)`. Token ID используется как номер строки этой таблицы. Это lookup, а не умножение ID как обычного числового признака.

In [6]:
vocab_size = len(vocabulary)
embedding_dim = 4
embedding = nn.Embedding(vocab_size, embedding_dim)

# Передаём token IDs в embedding-слой.
token_vectors = embedding(token_ids)

print("embedding weight:", embedding.weight.shape)
print("token IDs:", token_ids.shape)
print("token vectors:", token_vectors.shape)

print("embedding weight:", embedding.weight)
print("token IDs:", token_ids)
print("token vectors:", token_vectors)

embedding weight: torch.Size([5, 4])
token IDs: torch.Size([2, 3])
token vectors: torch.Size([2, 3, 4])
embedding weight: Parameter containing:
tensor([[ 0.3189, -0.4245,  0.3057, -0.7746],
        [-0.8371, -0.9224,  1.8113,  0.1606],
        [ 0.3672,  0.1754,  1.3852, -0.4459],
        [-1.2024,  0.7078, -1.0759,  0.5357],
        [ 1.1754,  0.5612, -0.4527, -0.7718]], requires_grad=True)
token IDs: tensor([[1, 2, 3],
        [1, 4, 3]])
token vectors: tensor([[[-0.8371, -0.9224,  1.8113,  0.1606],
         [ 0.3672,  0.1754,  1.3852, -0.4459],
         [-1.2024,  0.7078, -1.0759,  0.5357]],

        [[-0.8371, -0.9224,  1.8113,  0.1606],
         [ 1.1754,  0.5612, -0.4527, -0.7718],
         [-1.2024,  0.7078, -1.0759,  0.5357]]], grad_fn=<EmbeddingBackward0>)


**Предскажи и объясни:** почему форма меняется `(2, 3) → (2, 3, 4)`?

Исходный тензор содержит 2 последовательности по 3 token IDs. Каждый ID заменяется вектором длины 4, поэтому первые два измерения сохраняются, а в конце добавляется `embedding_dimension`: `(2, 3) → (2, 3, 4)`.

## 4. Один ID — одна строка embedding-матрицы

Проверь, что вектор токена с ID `1` буквально равен строке `embedding.weight[1]`.

In [7]:
first_sentence_first_token = token_vectors[0, 0]
row_for_token_id_1 = embedding.weight[1]

print(first_sentence_first_token)
print(row_for_token_id_1)
print(torch.equal(first_sentence_first_token, row_for_token_id_1))

tensor([-0.8371, -0.9224,  1.8113,  0.1606], grad_fn=<SelectBackward0>)
tensor([-0.8371, -0.9224,  1.8113,  0.1606], grad_fn=<SelectBackward0>)
True


**Вопрос:** почему одинаковый токен `я` в двух предложениях сначала получает одинаковый token embedding? Что позже может сделать его представления разными?

Оба вхождения токена `я` имеют один ID `1`, поэтому embedding lookup в обоих случаях выбирает одну и ту же строку `embedding.weight[1]`. При обучении эта общая строка изменяется, но остаётся общей для всех вхождений ID `1`. Представления двух позиций позже могут стать разными благодаря positional information и self-attention, который учитывает различающийся контекст.

## 5. Padding и sequence length

В одном batch тензор должен иметь прямоугольную форму. Более короткие последовательности дополняют специальным `<pad>` token ID. Actual `sequence_length` данного batch не обязательно равен максимальному context window модели.

In [8]:
padded_token_ids = torch.tensor([
    [1, 2, 3],
    [1, 4, 0],
])

padding_mask = padded_token_ids != vocabulary["<pad>"]
padded_vectors = embedding(padded_token_ids)

print("IDs shape:", padded_token_ids.shape)
print("vectors shape:", padded_vectors.shape)
print("padding mask:\n", padding_mask)

IDs shape: torch.Size([2, 3])
vectors shape: torch.Size([2, 3, 4])
padding mask:
 tensor([[ True,  True,  True],
        [ True,  True, False]])


**Наблюдение:** зачем позже attention понадобится padding mask?

Padding делает последовательности в batch одинаковыми по длине, но `<pad>` не является содержательной частью текста. Padding mask запрещает attention учитывать эти искусственные позиции при вычислении контекстных представлений.

## 6. Вопросы для собеседования

1. Чем tokenization отличается от embedding lookup?
2. Какова форма параметров `nn.Embedding(V, D)` и сколько их?
3. Что возвращает слой для входа token IDs формы `(B, T)`?
4. Почему token IDs нельзя считать числовой шкалой, где ID 100 «больше» ID 10?
5. Равен ли `sequence_length` максимальному context window модели?
6. Зачем нужны padding token и padding mask?

**Мои ответы:**

1. Tokenization разбивает текст на токены и преобразует их в целочисленные IDs. Embedding lookup использует каждый ID как индекс строки обучаемой embedding-матрицы и возвращает вектор.
2. `nn.Embedding(V, D)` содержит матрицу параметров формы `(V, D)`, всего $V \cdot D$ параметров.
3. Для token IDs формы `(B, T)` слой возвращает тензор embeddings формы `(B, T, D)`.
4. ID — только индекс строки в таблице; числовое расстояние и порядок между IDs не выражают смысловую близость токенов.
5. Нет. `sequence_length` — фактическая длина текущего входного тензора, а context window — максимально допустимая длина контекста модели. Обычно $T \leq context\_window$.
6. Padding token дополняет более короткие последовательности до общей длины batch. Padding mask сообщает attention, какие позиции являются искусственными и не должны влиять на результат.